In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pycot.reports import CommitmentsOfTraders

In [2]:
def visualize_cot_positions(contract_name, start_date=None, end_date=None, report_type='disaggregated'):
    """
    다양한 투자자 그룹의 포지션을 시각화하는 함수
    
    매개변수:
    - contract_name: 계약 이름 (예: 'WHEAT', 'CORN', 'GOLD')
    - start_date: 시작 날짜 (문자열: 'YYYY-MM-DD' 또는 datetime 객체)
    - end_date: 종료 날짜 (문자열: 'YYYY-MM-DD' 또는 datetime 객체)
    - report_type: 보고서 유형 ('legacy', 'disaggregated', 'financial')
    """
    # CoT 객체 생성
    cot = CommitmentsOfTraders(report_type=report_type)
    
    # 계약에 대한 보고서 가져오기
    df = cot.report(contract_name=contract_name)
    
    # 날짜 필터링
    if start_date:
        df = df[df['Date'] >= start_date]
    if end_date:
        df = df[df['Date'] <= end_date]
    
    # 날짜를 datetime으로 변환
    df['Date'] = pd.to_datetime(df['Date'])
    
    # 리포트 타입에 따라 투자자 그룹 컬럼 설정
    if report_type == 'legacy':
        position_columns = [
            'Commercial_Positions_Long_All', 
            'Commercial_Positions_Short_All',
            'NonCommercial_Positions_Long_All', 
            'NonCommercial_Positions_Short_All',
            'NonReportable_Positions_Long_All', 
            'NonReportable_Positions_Short_All'
        ]
        position_labels = ['Commercial Long', 'Commercial Short',
                         'Non-Commercial Long', 'Non-Commercial Short',
                         'Non-Reportable Long', 'Non-Reportable Short']
        
    elif report_type == 'disaggregated':
        position_columns = [
            'Producer_Merchant_Positions_Long_All', 
            'Producer_Merchant_Positions_Short_All',
            'Swap_Dealer_Positions_Long_All', 
            'Swap_Dealer_Positions_Short_All',
            'Managed_Money_Positions_Long_All', 
            'Managed_Money_Positions_Short_All',
            'Other_Reportable_Positions_Long_All', 
            'Other_Reportable_Positions_Short_All',
            'NonReportable_Positions_Long_All', 
            'NonReportable_Positions_Short_All'
        ]
        position_labels = ['Producer/Merchant Long', 'Producer/Merchant Short',
                         'Swap Dealer Long', 'Swap Dealer Short',
                         'Managed Money Long', 'Managed Money Short',
                         'Other Reportable Long', 'Other Reportable Short',
                         'Non-Reportable Long', 'Non-Reportable Short']
        
    elif report_type == 'financial':
        position_columns = [
            'Dealer_Positions_Long_All', 
            'Dealer_Positions_Short_All',
            'Asset_Manager_Positions_Long_All', 
            'Asset_Manager_Positions_Short_All',
            'Leveraged_Funds_Positions_Long_All', 
            'Leveraged_Funds_Positions_Short_All',
            'Other_Reportable_Positions_Long_All', 
            'Other_Reportable_Positions_Short_All',
            'NonReportable_Positions_Long_All', 
            'NonReportable_Positions_Short_All'
        ]
        position_labels = ['Dealer Long', 'Dealer Short',
                         'Asset Manager Long', 'Asset Manager Short',
                         'Leveraged Funds Long', 'Leveraged Funds Short',
                         'Other Reportable Long', 'Other Reportable Short',
                         'Non-Reportable Long', 'Non-Reportable Short']
    
    # 그래프 그리기
    plt.figure(figsize=(14, 8))
    
    for i, col in enumerate(position_columns):
        if col in df.columns:
            plt.plot(df['Date'], df[col], label=position_labels[i])
    
    plt.title(f'COT Positions for {contract_name} - {report_type.capitalize()} Report', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Number of Contracts', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best')
    
    # x축 날짜 포맷 설정
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.gcf().autofmt_xdate()
    
    plt.tight_layout()
    plt.show()
    
    # 순 포지션 시각화 (롱-숏)
    plt.figure(figsize=(14, 8))
    
    if report_type == 'legacy':
        net_positions = {
            'Commercial Net': df['Commercial_Positions_Long_All'] - df['Commercial_Positions_Short_All'],
            'Non-Commercial Net': df['NonCommercial_Positions_Long_All'] - df['NonCommercial_Positions_Short_All'],
            'Non-Reportable Net': df['NonReportable_Positions_Long_All'] - df['NonReportable_Positions_Short_All']
        }
    elif report_type == 'disaggregated':
        net_positions = {
            'Producer/Merchant Net': df['Producer_Merchant_Positions_Long_All'] - df['Producer_Merchant_Positions_Short_All'],
            'Swap Dealer Net': df['Swap_Dealer_Positions_Long_All'] - df['Swap_Dealer_Positions_Short_All'],
            'Managed Money Net': df['Managed_Money_Positions_Long_All'] - df['Managed_Money_Positions_Short_All'],
            'Other Reportable Net': df['Other_Reportable_Positions_Long_All'] - df['Other_Reportable_Positions_Short_All'],
            'Non-Reportable Net': df['NonReportable_Positions_Long_All'] - df['NonReportable_Positions_Short_All']
        }
    elif report_type == 'financial':
        net_positions = {
            'Dealer Net': df['Dealer_Positions_Long_All'] - df['Dealer_Positions_Short_All'],
            'Asset Manager Net': df['Asset_Manager_Positions_Long_All'] - df['Asset_Manager_Positions_Short_All'],
            'Leveraged Funds Net': df['Leveraged_Funds_Positions_Long_All'] - df['Leveraged_Funds_Positions_Short_All'],
            'Other Reportable Net': df['Other_Reportable_Positions_Long_All'] - df['Other_Reportable_Positions_Short_All'],
            'Non-Reportable Net': df['NonReportable_Positions_Long_All'] - df['NonReportable_Positions_Short_All']
        }
    
    for label, net_pos in net_positions.items():
        plt.plot(df['Date'], net_pos, label=label)
    
    plt.title(f'Net Positions for {contract_name} - {report_type.capitalize()} Report', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Net Positions (Long - Short)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best')
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # x축 날짜 포맷 설정
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.gcf().autofmt_xdate()
    
    plt.tight_layout()
    plt.show()

In [3]:
# 골드 포지션 시각화 (세분화 보고서)
visualize_cot_positions('GOLD', start_date='2020-01-01', end_date='2023-12-31', report_type='disaggregated_fut')

# 원유 포지션 시각화 (금융 보고서)
visualize_cot_positions('CRUDE OIL, LIGHT SWEET', start_date='2022-01-01', report_type='financial')

# 옥수수 포지션 시각화 (기존 레거시 보고서)
visualize_cot_positions('CORN', start_date='2021-01-01', end_date='2023-12-31', report_type='legacy')

31-Mar-25 23:12:01 - Extracting data for the disaggregated_fut report type


KeyError: 'Date'